In [4]:
"""
Study 2 — Evidence Path Sufficiency Analysis
Teacher (Path B) vs Student (Path A) Model Comparison

Items (8 total):
  Item 1: discovery → involves → skull
          A=Short/straight, B=Too much
  Item 2: Abdullah Hayayei → was hit during → training session
          A=Incomplete chain, B=More info but some irrelevant
  Item 3: legal challenge → against → building of student housing
          A=Insufficient, B=All required paths
  Item 4: Five people → connected to → murder
          A=Direct/straightforward, B=Too much
  Item 5: 850 appointment letters → contained → wrong leaflet
          A=Direct/sufficient, B=More information
  Item 6: source of Ebola outbreak → is → hollowed-out tree
          A=Needs understanding, B=Needs understanding
  Item 7: Kevin Nicholson → says → he will not receive any money
          A=Necessary info but needs understanding, B=Background included
  Item 8: Justin Rose → leads by → one shot
          A=Incomplete chain, B=Sufficient

Questions per item (7-point Likert):
  Q1 (path_a):   Path A sufficiently supports this summary claim
  Q2 (path_b):   Path B sufficiently supports this summary claim
  Q3 (equal):    Path A and Path B are equally sufficient
  Q4 (loss):     The shorter path loses important reasoning steps

Path assignment:
  Path A = Student model (Qwen3 8B LoRA) — shorter
  Path B = Teacher model (Claude Sonnet 4) — longer
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# ── Configuration ─────────────────────────────────────────────────────────────
FILE = "./Copy of Evidence Path Sufficiency Evaluation (Responses).xlsx"
OUT  = "./outputs/"

ITEMS = {
    1: {"triplet": "discovery → involves → skull",
        "path_a_char": "Short and straight",
        "path_b_char": "Too much information",
        "expected_winner": "A"},
    2: {"triplet": "Hayayei → was hit during → training",
        "path_a_char": "Incomplete chain",
        "path_b_char": "More info, some irrelevant",
        "expected_winner": "B"},
    3: {"triplet": "legal challenge → against → student housing",
        "path_a_char": "Insufficient",
        "path_b_char": "All required paths",
        "expected_winner": "B"},
    4: {"triplet": "Five people → connected to → murder",
        "path_a_char": "Direct and straightforward",
        "path_b_char": "Too much information",
        "expected_winner": "A"},
    5: {"triplet": "850 letters → contained → wrong leaflet",
        "path_a_char": "Direct and sufficient",
        "path_b_char": "More information",
        "expected_winner": "A"},
    6: {"triplet": "source of Ebola → is → hollowed-out tree",
        "path_a_char": "Needs understanding",
        "path_b_char": "Needs understanding",
        "expected_winner": "Equal"},
    7: {"triplet": "Kevin Nicholson → says → no money",
        "path_a_char": "Necessary info, needs understanding",
        "path_b_char": "Background included",
        "expected_winner": "B"},
    8: {"triplet": "Justin Rose → leads by → one shot",
        "path_a_char": "Incomplete chain",
        "path_b_char": "Sufficient information",
        "expected_winner": "B"},
}

COLOURS = {
    "path_a":  "#3b82f6",   # blue — student
    "path_b":  "#10b981",   # green — teacher
    "equal":   "#f59e0b",   # amber
    "loss":    "#ef4444",   # red
}

plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "figure.dpi":     150,
    "savefig.dpi":    300,
    "savefig.bbox":   "tight",
})

# ── Load and reshape ───────────────────────────────────────────────────────────
df_raw = pd.read_excel(FILE)
n_participants = len(df_raw)
print(f"Participants: {n_participants}")

# Build tidy dataframe
Q_NAMES = {
    "path_a": "Path A sufficiently supports this summary claim.",
    "path_b": "Path B sufficiently supports this summary claim.",
    "equal":  "Path A and Path B are equally sufficient explanations for this claim.",
    "loss":   "The shorter path loses important reasoning steps compared to the longer path.",
}

records = []
for item_num in range(1, 9):
    suffix = "" if item_num == 1 else f" {item_num}"
    for q_key, q_base in Q_NAMES.items():
        col = q_base + suffix
        if col in df_raw.columns:
            for p_idx, val in enumerate(df_raw[col]):
                records.append({
                    "participant": p_idx,
                    "item":        item_num,
                    "question":    q_key,
                    "score":       val,
                })

df = pd.DataFrame(records)
df["score"] = pd.to_numeric(df["score"], errors="coerce")

print(f"Total data points: {len(df)}")
print(f"Missing values: {df['score'].isna().sum()}")

# ── Overall statistics ─────────────────────────────────────────────────────────
print("\n" + "="*70)
print("OVERALL QUESTION STATISTICS (all items, all participants)")
print("="*70)
overall = df.groupby("question")["score"].agg(
    Mean="mean", SD="std", Median="median", Min="min", Max="max"
).round(3)
print(overall)

# ── Per-item statistics ────────────────────────────────────────────────────────
print("\n" + "="*70)
print("PER-ITEM STATISTICS")
print("="*70)
per_item = df.groupby(["item", "question"])["score"].agg(
    Mean="mean", SD="std"
).round(3)
print(per_item.to_string())

# ── Path A vs Path B: Primary comparison ──────────────────────────────────────
print("\n" + "="*70)
print("PATH A vs PATH B — PRIMARY COMPARISON")
print("="*70)

path_a_scores = df[df["question"] == "path_a"]["score"].dropna()
path_b_scores = df[df["question"] == "path_b"]["score"].dropna()

print(f"Path A (Student): Mean={path_a_scores.mean():.3f}, SD={path_a_scores.std():.3f}")
print(f"Path B (Teacher): Mean={path_b_scores.mean():.3f}, SD={path_b_scores.std():.3f}")

# Paired comparison per participant (mean across items)
path_a_per_p = df[df["question"] == "path_a"].groupby("participant")["score"].mean()
path_b_per_p = df[df["question"] == "path_b"].groupby("participant")["score"].mean()

if len(path_a_per_p) == len(path_b_per_p):
    stat, p = stats.wilcoxon(path_a_per_p.values, path_b_per_p.values)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"Wilcoxon signed-rank (paired per participant): W={stat:.1f}, p={p:.4f} {sig}")

# ── Per-item Path A vs B ───────────────────────────────────────────────────────
print("\n" + "="*70)
print("PER-ITEM PATH A vs B COMPARISON")
print("="*70)
print(f"{'Item':<8} {'A Mean':>8} {'A SD':>6} {'B Mean':>8} {'B SD':>6} {'Diff':>7} {'Winner':>8}")
print("-"*60)
for item_num in range(1, 9):
    a = df[(df["item"] == item_num) & (df["question"] == "path_a")]["score"].dropna()
    b = df[(df["item"] == item_num) & (df["question"] == "path_b")]["score"].dropna()
    diff = a.mean() - b.mean()
    winner = "A" if diff > 0.2 else "B" if diff < -0.2 else "≈Equal"
    print(f"Item {item_num:<3} {a.mean():>8.2f} {a.std():>6.2f} {b.mean():>8.2f} "
          f"{b.std():>6.2f} {diff:>+7.2f} {winner:>8}")

# ── Equality and information loss ──────────────────────────────────────────────
print("\n" + "="*70)
print("EQUALITY AND INFORMATION LOSS")
print("="*70)
equal_scores = df[df["question"] == "equal"]["score"].dropna()
loss_scores  = df[df["question"] == "loss"]["score"].dropna()
print(f"Equal sufficiency:    Mean={equal_scores.mean():.3f}, SD={equal_scores.std():.3f}")
print(f"Information loss:     Mean={loss_scores.mean():.3f}, SD={loss_scores.std():.3f}")
print(f"(Score > 4 = agree, Score < 4 = disagree, 4 = neutral)")

# ── Global questions ───────────────────────────────────────────────────────────
print("\n" + "="*70)
print("GLOBAL PERCEPTION QUESTIONS")
print("="*70)
global_cols = {
    "Overall shorter paths sufficient": "Overall, the shorter paths were sufficient to explain the summary facts.",
    "Overall longer paths more convincing": "Overall, the longer paths provided more convincing explanations.",
    "Task difficulty": "In general, how difficult was it to judge whether a path sufficiently supported a summary fact?",
}
for label, col in global_cols.items():
    if col in df_raw.columns:
        vals = pd.to_numeric(df_raw[col], errors="coerce").dropna()
        print(f"{label}: Mean={vals.mean():.2f}, SD={vals.std():.2f}, n={len(vals)}")

# ── Bias control check ─────────────────────────────────────────────────────────
print("\n" + "="*70)
print("BIAS CONTROL — 'I rated Path B higher because of model identity'")
print("="*70)
# Collect all bias control columns
bias_cols = [c for c in df_raw.columns if "rated Path B higher" in c]
bias_vals = []
for col in bias_cols:
    vals = pd.to_numeric(df_raw[col], errors="coerce").dropna()
    bias_vals.extend(vals.tolist())
if bias_vals:
    bias_arr = np.array(bias_vals)
    print(f"Mean={bias_arr.mean():.3f}, SD={bias_arr.std():.3f}")
    print(f"Proportion agreeing (>4): {(bias_arr > 4).mean():.1%}")
    print(f"Proportion neutral (=4): {(bias_arr == 4).mean():.1%}")
    print(f"Proportion disagreeing (<4): {(bias_arr < 4).mean():.1%}")
    print("(Low mean = participants rated on content, not model identity — good)")

# # ── Kruskal-Wallis across items ────────────────────────────────────────────────
# print("\n" + "="*70)
# print("KRUSKAL-WALLIS ACROSS ITEMS PER QUESTION")
# print("="*70)
# for q in ["path_a", "path_b", "equal", "loss"]:
#     groups = [df[(df["item"] == i) & (df["question"] == q)]["score"].dropna().values
#               for i in range(1, 9)]
#     groups = [g for g in groups if len(g) > 0]
#     if len(groups) > 1:
#         stat, p = stats.kruskal(*groups)
#         sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
#         print(f"  {q:10s}: H={stat:.3f}, p={p:.4f} {sig}")

# =========================
# FRIEDMAN TEST (CORRECT)
# =========================

print("\nFriedman test across items (per question):")

QUESTION_COLS = ["path_a", "path_b", "equal", "loss"]

for q in QUESTION_COLS:
    # Build participant × item matrix
    pivot = df[df["question"] == q].pivot_table(
        index="participant",
        columns="item",
        values="score"
    )

    # Drop incomplete participants (required)
    pivot = pivot.dropna()

    if pivot.shape[1] < 3:
        print(f"{q}: Not enough items for Friedman test")
        continue

    # Run Friedman
    stat, p = stats.friedmanchisquare(
        *[pivot[col] for col in pivot.columns]
    )

    # Effect size: Kendall's W
    n = pivot.shape[0]  # participants
    k = pivot.shape[1]  # items
    W = stat / (n * (k - 1))

    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

    print(f"{q:10s}: χ²={stat:.3f}, p={p:.4f} {sig}, W={W:.3f}")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURES
# ══════════════════════════════════════════════════════════════════════════════

# ── Figure 1: Path A vs B per item (grouped bar) ──────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

x = np.arange(8)
width = 0.28

a_means = [df[(df["item"]==i) & (df["question"]=="path_a")]["score"].mean() for i in range(1,9)]
b_means = [df[(df["item"]==i) & (df["question"]=="path_b")]["score"].mean() for i in range(1,9)]
a_sds   = [df[(df["item"]==i) & (df["question"]=="path_a")]["score"].std()  for i in range(1,9)]
b_sds   = [df[(df["item"]==i) & (df["question"]=="path_b")]["score"].std()  for i in range(1,9)]

bars_a = ax.bar(x - width/2, a_means, width, label="Path A (Student — LoRA)",
                color=COLOURS["path_a"], alpha=0.85, edgecolor="white")
bars_b = ax.bar(x + width/2, b_means, width, label="Path B (Teacher — Claude Sonnet)",
                color=COLOURS["path_b"], alpha=0.85, edgecolor="white")

ax.errorbar(x - width/2, a_means, yerr=a_sds, fmt="none",
            color="#1e293b", capsize=3, linewidth=1.2)
ax.errorbar(x + width/2, b_means, yerr=b_sds, fmt="none",
            color="#1e293b", capsize=3, linewidth=1.2)

# Value labels
for bar, mean in zip(bars_a, a_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f"{mean:.1f}", ha="center", va="bottom", fontsize=7.5)
for bar, mean in zip(bars_b, b_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f"{mean:.1f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels([f"I{i}" for i in range(1, 9)], fontsize=9)
ax.set_ylim(0, 8.5)
ax.set_yticks([1, 2, 3, 4, 5, 6, 7])
ax.axhline(4, color="#94a3b8", linewidth=0.8, linestyle="--", alpha=0.6)
ax.set_ylabel("Mean Sufficiency Score (1–7)", labelpad=8)
ax.set_xlabel("Item", labelpad=5)
ax.set_title("Study 2 — Path A (Student) vs Path B (Teacher) Sufficiency per Item\n"
             f"(n={n_participants} participants, error bars = ±1 SD)",
             fontweight="bold", pad=10)
ax.legend(fontsize=9, loc="lower right")
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUT + "study2_path_comparison.pdf")
plt.savefig(OUT + "study2_path_comparison.png", dpi=200)
plt.close()
print("\nSaved: study2_path_comparison.pdf")

# ── Figure 2: Equality and information loss per item ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, (q_key, colour, title) in zip(axes, [
    ("equal", COLOURS["equal"],  "Equal Sufficiency Rating\n(Q3: Path A = Path B)"),
    ("loss",  COLOURS["loss"],   "Information Loss Rating\n(Q4: Shorter path loses steps)"),
]):
    means = [df[(df["item"]==i) & (df["question"]==q_key)]["score"].mean() for i in range(1,9)]
    sds   = [df[(df["item"]==i) & (df["question"]==q_key)]["score"].std()  for i in range(1,9)]
    x = np.arange(8)

    bars = ax.bar(x, means, color=colour, alpha=0.82,
                  edgecolor="white", linewidth=0.8, zorder=2)
    ax.errorbar(x, means, yerr=sds, fmt="none",
                color="#1e293b", capsize=3, linewidth=1.2, zorder=3)

    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f"{mean:.1f}", ha="center", va="bottom", fontsize=8)

    ax.axhline(4, color="#94a3b8", linewidth=1, linestyle="--",
               alpha=0.7, label="Neutral (4)")
    ax.set_xticks(x)
    ax.set_xticklabels([f"I{i}" for i in range(1, 9)], fontsize=9)
    ax.set_ylim(0, 8.2)
    ax.set_yticks([1, 2, 3, 4, 5, 6, 7])
    ax.set_title(title, fontweight="bold", pad=8)
    ax.set_xlabel("Item", labelpad=5)
    ax.grid(axis="y", alpha=0.3, zorder=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Mean Likert Score (1–7)", labelpad=8)
fig.suptitle(f"Study 2 — Equivalence and Information Loss Ratings (n={n_participants})",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT + "study2_equality_loss.pdf")
plt.savefig(OUT + "study2_equality_loss.png", dpi=200)
plt.close()
print("Saved: study2_equality_loss.pdf")

# ── Figure 3: Overall heatmap — all 4 questions × 8 items ─────────────────────
q_order = ["path_a", "path_b", "equal", "loss"]
q_labels = ["Path A\n(Student)", "Path B\n(Teacher)", "Equal\nSufficiency", "Info\nLoss"]

heatmap_data = np.zeros((4, 8))
for qi, q in enumerate(q_order):
    for i in range(1, 9):
        val = df[(df["item"]==i) & (df["question"]==q)]["score"].mean()
        heatmap_data[qi, i-1] = val

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(heatmap_data, cmap="RdYlGn", vmin=1, vmax=7, aspect="auto")

ax.set_xticks(range(8))
ax.set_xticklabels([f"I{i}" for i in range(1, 9)], fontsize=9)
ax.set_yticks(range(4))
ax.set_yticklabels(q_labels, fontsize=9)

for i in range(4):
    for j in range(8):
        val = heatmap_data[i, j]
        text_col = "white" if val < 2.5 or val > 5.5 else "#1e293b"
        ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                fontsize=9, color=text_col, fontweight="bold")

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Mean Likert Score", fontsize=9)
cbar.set_ticks([1, 2, 3, 4, 5, 6, 7])

ax.set_title("Study 2 — All Questions × All Items Mean Scores\n"
             f"(n={n_participants} participants, 1=Strongly Disagree, 7=Strongly Agree)",
             fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(OUT + "study2_heatmap.pdf")
plt.savefig(OUT + "study2_heatmap.png", dpi=200)
plt.close()
print("Saved: study2_heatmap.pdf")

# ── LaTeX table ────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("LaTeX TABLE")
print("="*70)
print(r"\\begin{table}[h]")
print(r"\\centering")
print(r"\\small")
print(r"\\begin{tabular}{lcccccccc}")
print(r"\\hline")
print(r"\\textbf{Item} & \\multicolumn{2}{c}{\\textbf{Path A (Student)}} & "
      r"\\multicolumn{2}{c}{\\textbf{Path B (Teacher)}} & "
      r"\\multicolumn{2}{c}{\\textbf{Equal}} & "
      r"\\multicolumn{2}{c}{\\textbf{Info Loss}} \\\\")
print(r" & M & SD & M & SD & M & SD & M & SD \\\\")
print(r"\\hline")

for item_num in range(1, 9):
    row = []
    for q in ["path_a", "path_b", "equal", "loss"]:
        scores = df[(df["item"]==item_num) & (df["question"]==q)]["score"].dropna()
        row.append(f"{scores.mean():.2f}")
        row.append(f"{scores.std():.2f}")
    print(f"Item {item_num:02d} & " + " & ".join(row) + r" \\\\")

print(r"\\hline")
# Overall row
row = []
for q in ["path_a", "path_b", "equal", "loss"]:
    scores = df[df["question"]==q]["score"].dropna()
    row.append(f"\\textbf{{{scores.mean():.2f}}}")
    row.append(f"{scores.std():.2f}")
print(r"\\textbf{Overall} & " + " & ".join(row) + r" \\\\")
print(r"\\hline")
print(r"\\end{tabular}")
print(r"\\caption{Study 2 Likert results per item (n=" + str(n_participants) + r"). "
      r"Path A = student model (Qwen3 8B LoRA); Path B = teacher model (Claude Sonnet 4). "
      r"Equal = equal sufficiency rating; Info Loss = shorter path loses reasoning steps. "
      r"M = mean, SD = standard deviation.}")
print(r"\\label{tab:study2_results}")
print(r"\\end{table}")

print(f"\n✓ Analysis complete. {n_participants} participants. Figures saved to outputs/")
print("\nNOTE: Re-run this script after adding remaining participants.")

Participants: 10
Total data points: 320
Missing values: 0

OVERALL QUESTION STATISTICS (all items, all participants)
           Mean     SD  Median  Min  Max
question                                
equal     4.775  1.615     5.0    1    7
loss      4.162  2.003     4.0    1    7
path_a    5.025  1.449     5.0    1    7
path_b    5.762  1.082     6.0    2    7

PER-ITEM STATISTICS
               Mean     SD
item question             
1    equal      4.4  2.119
     loss       4.3  2.359
     path_a     5.4  1.350
     path_b     6.5  0.527
2    equal      5.3  0.949
     loss       3.7  2.003
     path_a     5.3  1.059
     path_b     5.7  1.059
3    equal      3.8  1.619
     loss       4.5  1.434
     path_a     4.4  1.506
     path_b     5.3  1.337
4    equal      4.4  1.776
     loss       4.9  1.853
     path_a     4.4  1.506
     path_b     5.7  0.675
5    equal      5.7  0.823
     loss       3.9  2.079
     path_a     5.9  0.738
     path_b     6.3  0.823
6    equal      5.4  1